In [46]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
import json

path = "/content/drive/MyDrive/dataset/sirene_propre.json"

with open(path, "r", encoding="utf-8") as f:
    entreprises = json.load(f)

len(entreprises), entreprises[0]


(50,
 {'siren': '306121450',
  'nic': '00010',
  'siret': '30612145000010',
  'dateCreationEtablissement': None,
  'etablissementSiege': True,
  'etatAdministratifUniteLegale': 'C',
  'dateCreationUniteLegale': '1900-01-01',
  'denominationUniteLegale': None,
  'activitePrincipaleUniteLegale': '55.60',
  'nomenclatureActivitePrincipaleUniteLegale': 'NAP',
  'categorieEntreprise': None,
  'adresseEtablissement': '8 RUE BORGNIAT, 10140 VENDEUVRE-SUR-BARSE FRANCE',
  'complementAdresseEtablissement': None,
  'numeroVoieEtablissement': '8',
  'indiceRepetitionEtablissement': None,
  'typeVoieEtablissement': 'RUE',
  'libelleVoieEtablissement': 'BORGNIAT',
  'codePostalEtablissement': '10140',
  'libelleCommuneEtablissement': 'VENDEUVRE-SUR-BARSE',
  'metadata': {'pipeline_processing_date': '2026-03-17T11:27:02.277031'}})

In [48]:
!pip install faker


In [49]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker("fr_FR")

from datetime import datetime, timedelta
import random

def random_date_after_creation(company, min_year=1990):
    # 1. Récupérer la date de création réelle
    creation_str = company.get("dateCreationUniteLegale") or company.get("dateCreationEtablissement")

    # Si aucune date → fallback
    if not creation_str:
        creation_date = datetime(min_year, 1, 1)
    else:
        try:
            creation_date = datetime.strptime(creation_str, "%Y-%m-%d")
        except:
            creation_date = datetime(min_year, 1, 1)

    # 2. On ne veut pas de dates trop anciennes
    if creation_date.year < min_year:
        creation_date = datetime(min_year, 1, 1)

    # 3. Date max = aujourd’hui
    end_date = datetime.now()

    # 4. Générer une date aléatoire entre création et aujourd’hui
    delta = end_date - creation_date
    random_days = random.randint(0, delta.days)
    random_date = creation_date + timedelta(days=random_days)

    return random_date.strftime("%Y-%m-%d")


def generate_urssaf_certificate(company):
    siren = company.get("siren")
    siret = company.get("siret")
    adresse = company.get("adresseEtablissement") or (
        f"{company.get('numeroVoieEtablissement','')} "
        f"{company.get('typeVoieEtablissement','')} "
        f"{company.get('libelleVoieEtablissement','')}, "
        f"{company.get('codePostalEtablissement','')} "
        f"{company.get('libelleCommuneEtablissement','')}"
    )

    return {
        "siren": siren,
        "siret": siret,
        "social_security": fake.ssn(),
        "internal_identifier": f"URSSAF-{random.randint(100000,999999)}",
        "security_code": fake.bothify("??##-??##").upper(),
        "created_at": random_date_after_creation(company),

        "place_at": fake.city(),
        "activity_address": adresse
    }




In [50]:
!pip install reportlab


In [62]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.graphics.barcode import qr
from reportlab.graphics.shapes import Drawing
from reportlab.graphics import renderPDF

def urssaf_to_pdf_reportlab(att, filename):
    c = canvas.Canvas(filename, pagesize=A4)
    width, height = A4

    # ---------------------------------------------------------
    # 1) BANDEAU / LOGO URSSAF
    # ---------------------------------------------------------
    c.setFillColorRGB(0, 0.35, 0.65)
    c.rect(0, height - 25 * mm, width, 18 * mm, fill=1, stroke=0)

    # Logo à gauche
    c.setFillColor(colors.white)
    c.setFont("Helvetica-Bold", 20)
    c.drawString(15 * mm, height - 14 * mm, "Urssaf")
    c.setFont("Helvetica", 9)
    c.drawString(15 * mm, height - 19 * mm, "Au service de la protection sociale")

    # ---------------------------------------------------------
    # 2) INFORMATIONS URSSAF EN HAUT À DROITE (DANS LE BANDEAU)
    # ---------------------------------------------------------
    info_x = width - 70 * mm   # position à droite
    info_y = height - 14 * mm

    c.setFont("Helvetica-Bold", 9)
    c.drawString(info_x, info_y, "Informations URSSAF :")
    info_y -= 4.5 * mm

    c.setFont("Helvetica", 8)
    c.drawString(info_x, info_y, "Téléphone : 3698")
    info_y -= 4.5 * mm
    c.drawString(info_x, info_y, "Site : www.urssaf.fr")
    info_y -= 4.5 * mm
    c.drawString(info_x, info_y, f"Code sécurité : {att['security_code']}")
    info_y -= 4.5 * mm
    c.drawString(info_x, info_y, f"Identifiant : {att['internal_identifier']}")

    c.setFillColor(colors.black)

    # ---------------------------------------------------------
    # 3) LIGNE VERTICALE À GAUCHE
    # ---------------------------------------------------------
    c.setStrokeColorRGB(0, 0.35, 0.65)
    c.setLineWidth(3)
    c.line(12 * mm, 20 * mm, 12 * mm, height - 20 * mm)

    # ---------------------------------------------------------
    # 4) TITRE CENTRÉ
    # ---------------------------------------------------------
    c.setFont("Helvetica-Bold", 15)
    c.drawCentredString(width / 2, height - 55 * mm, "ATTESTATION DE VIGILANCE")

    # ---------------------------------------------------------
    # 5) CORPS DU DOCUMENT (DÉCALÉ À DROITE)
    # ---------------------------------------------------------
    margin_left = 45 * mm
    y = height - 75 * mm

    c.setFont("Helvetica-Bold", 10)
    c.drawString(margin_left, y, "Cadre légal :")
    y -= 5 * mm

    c.setFont("Helvetica", 8.5)
    c.drawString(margin_left, y, "Articles L.8222-1 à L.8222-3 et D.8222-5 du Code du Travail.")
    y -= 4.5 * mm
    c.drawString(margin_left, y, "La validité de ce document peut être vérifiée sur le site de l'URSSAF.")
    y -= 8 * mm

    # --- Informations entreprise ---
    c.setFont("Helvetica-Bold", 10)
    c.drawString(margin_left, y, "Informations de l'entreprise :")
    y -= 6 * mm

    c.setFont("Helvetica", 8.5)
    c.drawString(margin_left, y, f"SIREN : {att['siren']}")
    y -= 4.5 * mm
    c.drawString(margin_left, y, f"SIRET : {att['siret']}")
    y -= 4.5 * mm
    c.drawString(margin_left, y, f"N° Sécurité Sociale : {att['social_security']}")
    y -= 6 * mm

    c.drawString(margin_left, y, "Adresse d'activité :")
    y -= 4.5 * mm
    for line in att["activity_address"].split(","):
        c.drawString(margin_left + 6 * mm, y, line.strip())
        y -= 4.5 * mm

    y -= 8 * mm

    # --- Certification ---
    c.setFont("Helvetica-Bold", 10)
    c.drawString(margin_left, y, "Certification :")
    y -= 6 * mm

    c.setFont("Helvetica", 8.5)
    c.drawString(margin_left, y,
        "L'URSSAF certifie que l'entreprise mentionnée ci-dessus est à jour de ses obligations.")
    y -= 12 * mm

    # ---------------------------------------------------------
    # 6) DATE & SIGNATURE
    # ---------------------------------------------------------
    c.setFont("Helvetica", 8.5)
    c.drawString(margin_left, y, f"Fait à : {att['place_at']}")
    y -= 4.5 * mm
    c.drawString(margin_left, y, f"Le : {att['created_at']}")
    y -= 8 * mm

    c.setFont("Helvetica-Oblique", 10)
    c.drawString(margin_left, y, "Le Directeur,")
    y -= 7 * mm

    c.setFont("Helvetica", 9)
    c.drawString(margin_left + 5 * mm, y, fake.name())

    # ---------------------------------------------------------
    # 7) QR CODE
    # ---------------------------------------------------------
    qr_data = f"SIREN:{att['siren']}|SIRET:{att['siret']}|CODE:{att['security_code']}"
    qr_code = qr.QrCodeWidget(qr_data)
    bounds = qr_code.getBounds()
    size = 22 * mm
    w = bounds[2] - bounds[0]
    h = bounds[3] - bounds[1]
    d = Drawing(size, size, transform=[size / w, 0, 0, size / h, 0, 0])
    d.add(qr_code)

    renderPDF.draw(d, c, width - 40 * mm, 25 * mm)

    c.save()


In [63]:
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/dataset/attestations_urssaf")
output_dir.mkdir(parents=True, exist_ok=True)

for i, company in enumerate(entreprises):
    att = generate_urssaf_certificate(company)
    pdf_path = output_dir / f"urssaf_{i:03d}.pdf"
    urssaf_to_pdf_reportlab(att, str(pdf_path))